# 04. Поведенческая сегментация

> **Краткое резюме**: Кластеризуем клиентов по поведенческому профилю (обороты, контрагенты, направленность, динамика). Пересечение с модельным сегментом показывает, насколько поведение совпадает с формальной классификацией. Результат — каждый клиент получает `behavioral_segment`.

**Предпосылки**: `03_feature_engineering.ipynb` выполнен.

---

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import MinMaxScaler

OUTPUT_DIR = os.path.abspath("./data")

In [ ]:
# =====================================================
# ПАРАМЕТРЫ
# =====================================================

K_MIN = 3  # Минимальное число кластеров
K_MAX = 15  # Максимальное (расширено для экспертного анализа)
# None = автоподбор по silhouette + Calinski-Harabasz
# Число = фиксированный k (для debug / быстрого прогона)
# Рекомендация: K_OVERRIDE = 12 для детального анализа экспертом
K_OVERRIDE = None

# Порог обрезки выбросов в нормализованном пространстве (±σ)
# Предотвращает формирование кластеров из 1–2 клиентов
CLIP_SIGMA = 3.0

---
## 1. Загрузка данных

In [ ]:
X = pd.read_parquet(os.path.join(OUTPUT_DIR, "feature_matrix.parquet"))
full_df = pd.read_parquet(os.path.join(OUTPUT_DIR, "full_client_data.parquet"))

with open(os.path.join(OUTPUT_DIR, "feature_meta.pkl"), "rb") as f:
    meta = pickle.load(f)

print(f"Feature matrix: {X.shape}")
print(f"Full data: {full_df.shape}")

---
## 1.5. Винзоризация выбросов

Клиенты-хабы с аномально высоким числом транзакций или контрагентов (например, 3000+ tx, 2500+ counterparties)
формируют отдельный кластер из 1 человека — это артефакт, а не бизнес-сегмент.

Обрезаем значения за пределами ±{CLIP_SIGMA}σ в нормализованном пространстве.
Клиенты **остаются** в данных, просто их экстремальные координаты приводятся к разумному диапазону.

In [ ]:
# Обрезаем значения в нормализованном пространстве
X_clipped = X.clip(lower=-CLIP_SIGMA, upper=CLIP_SIGMA)

n_outlier_vals = (X.abs() > CLIP_SIGMA).sum().sum()
n_outlier_clients = (X.abs() > CLIP_SIGMA).any(axis=1).sum()
print(f"Clipped {n_outlier_vals:,} values exceeding ±{CLIP_SIGMA}σ")
print(f"Affected clients: {n_outlier_clients:,} ({n_outlier_clients / len(X) * 100:.1f}%)")

# Показываем топ-5 клиентов по «экстремальности» (только числовые столбцы)
extreme_score = X.select_dtypes(include="number").abs().max(axis=1)
print("\nТоп-5 самых экстремальных клиентов (до обрезки):")
print(extreme_score.nlargest(5))

In [ ]:
X_vals = X_clipped.values  # используем обрезанные данные

if K_OVERRIDE is not None:
    best_k = K_OVERRIDE
    print(f"Using override k={best_k}")
    res_df = None
else:
    max_k = min(K_MAX, len(X) - 1)
    results = []
    for k in range(K_MIN, max_k + 1):
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_vals)
        if len(set(labels)) < 2:
            continue
        sil = silhouette_score(X_vals, labels, sample_size=min(10_000, len(X)))
        ch = calinski_harabasz_score(X_vals, labels)
        db = davies_bouldin_score(X_vals, labels)
        results.append(
            {
                "k": k,
                "silhouette": sil,
                "calinski_harabasz": ch,
                "davies_bouldin": db,
                "inertia": km.inertia_,
            }
        )
        print(f"  k={k}: silhouette={sil:.4f}  CH={ch:.1f}  DB={db:.4f}")

    res_df = pd.DataFrame(results)

    def minmax(s: pd.Series) -> pd.Series:
        rng = s.max() - s.min()
        return (s - s.min()) / rng if rng > 0 else pd.Series(0.5, index=s.index)

    res_df["score"] = (
        minmax(res_df["silhouette"])
        + minmax(res_df["calinski_harabasz"])
        - minmax(res_df["davies_bouldin"])
    )
    best_k = int(res_df.loc[res_df["score"].idxmax(), "k"])
    best_row = res_df.loc[res_df["score"].idxmax()]
    print(
        f"\nBest k={best_k}  (silhouette={best_row['silhouette']:.4f},"
        f"  CH={best_row['calinski_harabasz']:.1f},"
        f"  DB={best_row['davies_bouldin']:.4f})"
    )
    display(res_df.round(4))

In [ ]:
if res_df is not None and len(res_df) > 1:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(res_df["k"], res_df["silhouette"], "o-", linewidth=2)
    axes[0, 0].axvline(best_k, color="red", linestyle="--", label=f"best k={best_k}")
    axes[0, 0].set_title("Silhouette Score (↑ лучше)")
    axes[0, 0].legend()

    axes[0, 1].plot(res_df["k"], res_df["inertia"], "o-", linewidth=2, color="orange")
    axes[0, 1].axvline(best_k, color="red", linestyle="--")
    axes[0, 1].set_title("Inertia / Elbow (ищем «локоть»)")

    axes[1, 0].plot(res_df["k"], res_df["calinski_harabasz"], "o-", linewidth=2, color="green")
    axes[1, 0].axvline(best_k, color="red", linestyle="--")
    axes[1, 0].set_title("Calinski-Harabasz (↑ лучше)")

    axes[1, 1].plot(res_df["k"], res_df["davies_bouldin"], "o-", linewidth=2, color="purple")
    axes[1, 1].axvline(best_k, color="red", linestyle="--")
    axes[1, 1].set_title("Davies-Bouldin (↓ лучше)")

    for ax in axes.flat:
        ax.set_xlabel("k")
    plt.suptitle("Выбор оптимального k кластеров", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

---
## 3. Финальная кластеризация

In [ ]:
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
full_df["behavioral_segment"] = km_final.fit_predict(X_vals)

print(f"Segments assigned (k={best_k}):")
print(full_df["behavioral_segment"].value_counts().sort_index())

---
## 4. Профили сегментов

Средние значения ключевых поведенческих метрик по сегментам.

In [ ]:
PROFILE_COLS = [
    "total_amount",
    "tx_count",
    "unique_counterparties",
    "direction_ratio",
    "active_months",
    "amount_growth",
    "cp_growth",
]
for col in ["avg_balance", "product_count", "avg_monthly_amount", "tx_amount_cv"]:
    if col in full_df.columns:
        PROFILE_COLS.append(col)

segment_profiles = full_df.groupby("behavioral_segment")[PROFILE_COLS].mean()
print("Segment profiles (mean):")
display(segment_profiles.round(2))

In [ ]:
# ruff: noqa: F821
# Пороги на основе реального распределения всей популяции.
_label_cols = [
    c
    for c in ["tx_count", "unique_counterparties", "total_amount", "active_months", "amount_growth"]
    if c in full_df.columns
]
_q25 = full_df[_label_cols].quantile(0.25)
_q75 = full_df[_label_cols].quantile(0.75)
_q90 = full_df[_label_cols].quantile(0.90)
_q95 = full_df[_label_cols].quantile(0.95)
_med = full_df[_label_cols].median()


def _segment_levels(profile: pd.Series) -> dict[str, bool]:
    """Вычисляет уровни метрик для профиля сегмента."""
    tx = profile.get("tx_count", 0)
    cp = profile.get("unique_counterparties", 0)
    amt = profile.get("total_amount", 0)
    act = profile.get("active_months", 0)
    grow = profile.get("amount_growth", 0)
    dr = profile.get("direction_ratio", 0.5)
    return {
        "mega_tx": tx >= _q95.get("tx_count", 1e9),
        "mega_cp": cp >= _q95.get("unique_counterparties", 1e9),
        "high_tx": tx >= _q75.get("tx_count", 1e9),
        "high_cp": cp >= _q75.get("unique_counterparties", 1e9),
        "high_amt": amt >= _q75.get("total_amount", 1e9),
        "growing": grow >= _q75.get("amount_growth", 1e9),
        "active": act >= _med.get("active_months", 0),
        "declining": grow < _q25.get("amount_growth", -1e9),
        "low_tx": tx < _q25.get("tx_count", 0),
        "low_cp": cp < _q25.get("unique_counterparties", 0),
        "payer": dr >= 0.70,
        "receiver": dr <= 0.30,
        "balanced": 0.30 <= dr <= 0.70,
    }


def _ref_potential(lvl: dict[str, bool]) -> tuple[int, str]:
    """Composite reference potential score (0-100) + label."""
    score = 0
    score += 30 if (lvl["mega_tx"] or lvl["mega_cp"]) else 0
    score += 20 if (lvl["high_tx"] or lvl["high_cp"]) else 0
    score -= 10 if (lvl["low_tx"] and lvl["low_cp"]) else 0
    score += 25 if lvl["growing"] else 0
    score -= 15 if lvl["declining"] else 0
    score += 15 if lvl["active"] else 0
    score += 15 if lvl["high_amt"] else 0
    score += 15 if lvl["balanced"] else 0
    labels = {60: "Высокий", 30: "Средний", 0: "Низкий"}
    for threshold, label in labels.items():
        if score >= threshold:
            return score, label
    return score, "Не подходит"


# Правила именования: первое совпадение выигрывает
_RULES: list[tuple] = [
    (lambda lv: lv["mega_tx"] and lv["mega_cp"], "Мега-хаб (системообразующий)"),
    (
        lambda lv: lv["high_tx"] and lv["high_cp"] and lv["growing"],
        "Растущий хаб (активность + рост)",
    ),
    (lambda lv: lv["high_tx"] and lv["high_cp"], "Хаб (много контрагентов, высокая активность)"),
    (
        lambda lv: lv["growing"] and (lv["high_tx"] or lv["high_cp"]),
        "Растущий активный (рост + сеть)",
    ),
    (lambda lv: lv["growing"], "Растущий (агрессивный рост оборота)"),
    (lambda lv: lv["payer"] and lv["declining"], "Плательщик (расходы, оборот падает)"),
    (lambda lv: lv["payer"], "Плательщик (преимущественно расходы)"),
    (lambda lv: lv["receiver"] and lv["declining"], "Получатель (входящие, оборот падает)"),
    (lambda lv: lv["receiver"], "Получатель (преимущественно входящие)"),
    (lambda lv: lv["high_amt"] and lv["active"], "Крупный стабильный (объём + регулярность)"),
    (lambda lv: lv["low_tx"] and lv["low_cp"] and lv["declining"], "Неактивный / затухающий"),
    (lambda lv: lv["low_tx"] and lv["low_cp"], "Малоактивный (низкие показатели)"),
]


def auto_label_segment(profile: pd.Series) -> tuple[str, str]:
    """Бизнес-название + потенциал для эталона."""
    lvl = _segment_levels(profile)
    _, potential = _ref_potential(lvl)
    name = "Средний / смешанный профиль"
    for cond, label in _RULES:
        if cond(lvl):
            name = label
            break
    return name, potential


segment_labels = {}
segment_potentials = {}
for seg_id, row in segment_profiles.iterrows():
    name, potential = auto_label_segment(row)
    segment_labels[seg_id] = name
    segment_potentials[seg_id] = potential

full_df["segment_label"] = full_df["behavioral_segment"].map(segment_labels)

print("Маркировка сегментов:")
print("─" * 75)
for seg_id, label in sorted(segment_labels.items()):
    n = (full_df["behavioral_segment"] == seg_id).sum()
    pct = n / len(full_df) * 100
    pot = segment_potentials[seg_id]
    pot_icon = {"Высокий": "✅", "Средний": "⚠️ ", "Низкий": "❌", "Не подходит": "⛔"}.get(pot, "?")
    print(f"  Сег. {seg_id} | {n:>5,} ({pct:4.1f}%) | {pot_icon} Потенциал: {pot:12s} | {label}")
print("─" * 75)
print("\nПотенциал рассчитан по 6 измерениям:")
print("  tx_count, counterparties, growth, active_months, amount, direction")

---
## 4.5. Сводная таблица сегментов для экспертной оценки

Таблица содержит все агрегаты, по которым определялись сегменты, в формате
удобном для анализа. Экспортируется в CSV для передачи эксперту.

In [ ]:
# ruff: noqa: F821
# ── Сводная таблица сегментов ────────────────────────────────────────────────
# Все метрики, использованные для кластеризации + бизнес-метки + потенциал.
# Экспорт в CSV для передачи эксперту.

# Расширенный набор метрик для профилирования
_EXTENDED_COLS = [
    "total_amount",
    "total_outflow",
    "total_inflow",
    "tx_count",
    "avg_tx_amount",
    "unique_counterparties",
    "unique_payees",
    "unique_payers",
    "direction_ratio",
    "active_months",
    "amount_growth",
    "cp_growth",
    "tx_per_month",
    "cp_per_month",
    "payee_ratio",
    "payer_ratio",
    "avg_monthly_amount",
    "tx_amount_cv",
    "amt_per_cp",
]
# Граф-метрики (если есть)
for _gc in [
    "pagerank",
    "betweenness",
    "clustering_coef",
    "flow_through_ratio",
    "in_degree",
    "out_degree",
    "okved_diversity_entropy",
    "top_k_concentration",
    "network_influence",
]:
    if _gc in full_df.columns:
        _EXTENDED_COLS.append(_gc)
# Производные (если есть)
for _dc in [
    "herfindahl_index",
    "monthly_volatility",
    "counterparty_retention",
    "balance_turnover_ratio",
    "growth_acceleration",
]:
    if _dc in full_df.columns:
        _EXTENDED_COLS.append(_dc)
# Балансовые (если есть)
for _bc in ["avg_balance", "avg_balance_30d", "product_count"]:
    if _bc in full_df.columns:
        _EXTENDED_COLS.append(_bc)

_available = [c for c in _EXTENDED_COLS if c in full_df.columns]

# Агрегаты по сегментам: mean, median, std
seg_mean = full_df.groupby("behavioral_segment")[_available].mean()
seg_median = full_df.groupby("behavioral_segment")[_available].median()
seg_std = full_df.groupby("behavioral_segment")[_available].std()

# Общие медианы для сравнения
_overall_med = full_df[_available].median()

# Собираем итоговую таблицу
rows = []
for seg_id in sorted(full_df["behavioral_segment"].unique()):
    n = int((full_df["behavioral_segment"] == seg_id).sum())
    pct = n / len(full_df) * 100
    label = segment_labels.get(seg_id, "?")
    potential = segment_potentials.get(seg_id, "?")
    row = {
        "segment_id": seg_id,
        "label": label,
        "potential": potential,
        "n_clients": n,
        "pct_of_total": round(pct, 1),
    }
    # Mean для каждой метрики
    for col in _available:
        row[f"{col}_mean"] = round(seg_mean.loc[seg_id, col], 4)
        row[f"{col}_median"] = round(seg_median.loc[seg_id, col], 4)
        # Отношение к общей медиане
        med_val = _overall_med[col]
        if med_val != 0 and not pd.isna(med_val):
            row[f"{col}_vs_median"] = round(seg_mean.loc[seg_id, col] / med_val, 2)
        else:
            row[f"{col}_vs_median"] = None

    rows.append(row)

# Добавляем строку "ВСЯ БАЗА" для сравнения
overall_row = {
    "segment_id": "ALL",
    "label": "ВСЯ БАЗА (reference)",
    "potential": "-",
    "n_clients": len(full_df),
    "pct_of_total": 100.0,
}
for col in _available:
    overall_row[f"{col}_mean"] = round(full_df[col].mean(), 4)
    overall_row[f"{col}_median"] = round(full_df[col].median(), 4)
    overall_row[f"{col}_vs_median"] = 1.0

rows.append(overall_row)

segment_analysis_df = pd.DataFrame(rows)

# ── Краткая таблица для отображения в ноутбуке ───────────────────────────────
_display_metrics = [
    "total_amount",
    "tx_count",
    "unique_counterparties",
    "direction_ratio",
    "active_months",
    "amount_growth",
]
_display_cols = ["segment_id", "label", "potential", "n_clients", "pct_of_total"]
for m in _display_metrics:
    if f"{m}_mean" in segment_analysis_df.columns:
        _display_cols.append(f"{m}_mean")
        _display_cols.append(f"{m}_vs_median")

print("Сводная таблица сегментов (ключевые метрики):")
print("vs_median: отношение среднего сегмента к медиане всей базы")
print("  >1.0 = выше медианы, <1.0 = ниже медианы")
display(
    segment_analysis_df[_display_cols]
    .style.format(precision=2)
    .background_gradient(
        subset=[c for c in _display_cols if c.endswith("_vs_median")],
        cmap="RdYlGn",
        vmin=0.3,
        vmax=2.0,
    )
)

# ── Экспорт полной таблицы в CSV ─────────────────────────────────────────────
csv_path = os.path.join(OUTPUT_DIR, "segment_profiles_for_review.csv")
segment_analysis_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print()
print(
    f"Полная таблица ({len(segment_analysis_df)} строк, "
    f"{len(segment_analysis_df.columns)} колонок) сохранена:"
)
print(f"  {csv_path}")
print()
print("Откройте CSV в Excel для экспертной оценки.")
print("Колонки *_mean — среднее по сегменту.")
print("Колонки *_median — медиана по сегменту.")
print("Колонки *_vs_median — отношение среднего к медиане ВСЕЙ базы.")

In [ ]:
# ruff: noqa: F821
norm_profiles = pd.DataFrame(
    MinMaxScaler().fit_transform(segment_profiles),
    index=segment_profiles.index,
    columns=segment_profiles.columns,
)

labels = [c[:14] for c in norm_profiles.columns]
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
colors = plt.cm.tab10.colors

for i, (seg_id, row) in enumerate(norm_profiles.iterrows()):
    vals = row.tolist() + [row.tolist()[0]]
    lbl = f"Seg {seg_id}: {segment_labels.get(seg_id, '')}"
    ax.plot(angles, vals, "o-", label=lbl, color=colors[i % len(colors)], linewidth=2)
    ax.fill(angles, vals, alpha=0.08, color=colors[i % len(colors)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=9)
ax.set_title("Профили сегментов (нормализовано)", fontsize=13, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.55, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ruff: noqa: F821
# ── UMAP: 2D визуализация кластеров ─────────────────────────────────────────
# UMAP сохраняет локальную структуру данных лучше PCA, особенно для
# поведенческих признаков с нелинейными зависимостями.
#
# Ошибка "AttributeError: 'float' object has no attribute 'shape'" в numpy.average
# — баг совместимости umap-learn < 0.5.6 с numpy >= 2.0.
# Лечится: uv pip install "umap-learn>=0.5.6"
try:
    import warnings

    import umap

    print("Fitting UMAP (cosine, 2D)…")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        reducer = umap.UMAP(
            n_components=2,
            metric="cosine",
            n_neighbors=30,
            min_dist=0.1,
            random_state=42,
        )
        X_2d = reducer.fit_transform(X_vals)

    seg_ids = sorted(full_df["behavioral_segment"].unique())
    seg_assignment = full_df.loc[X.index, "behavioral_segment"].values
    colors = plt.cm.tab10.colors

    fig, ax = plt.subplots(figsize=(13, 8))
    for i, seg_id in enumerate(seg_ids):
        mask = seg_assignment == seg_id
        lbl = f"Seg {seg_id}: {segment_labels.get(seg_id, '')[:30]}"
        ax.scatter(
            X_2d[mask, 0],
            X_2d[mask, 1],
            c=[colors[i % len(colors)]],
            label=lbl,
            alpha=0.4,
            s=6,
            rasterized=True,
        )
        # Centroid marker
        cx, cy = X_2d[mask, 0].mean(), X_2d[mask, 1].mean()
        ax.scatter(
            cx, cy, c=[colors[i % len(colors)]], s=200, marker="*", edgecolors="black", zorder=5
        )
        ax.annotate(
            f"Seg {seg_id}",
            (cx, cy),
            fontsize=9,
            fontweight="bold",
            textcoords="offset points",
            xytext=(6, 6),
        )

    ax.set_title("UMAP — клиенты в 2D (cosine distance)\n★ = центроид сегмента", fontsize=13)
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
    plt.tight_layout()
    plt.show()
    print("✅ UMAP: хорошо разделённые кластеры означают устойчивую сегментацию")

except ImportError:
    print("⚠️  umap-learn не установлен: uv pip install umap-learn")
except AttributeError as e:
    if "shape" in str(e):
        print("⚠️  Несовместимость umap-learn с текущей версией numpy.")
        print("   Исправление: uv pip install 'umap-learn>=0.5.6'")
        print(f"   Подробнее: {e}")
    else:
        raise

In [ ]:
# ruff: noqa: F821
# ── HDBSCAN vs KMeans: сравнение методов кластеризации ──────────────────────
# HDBSCAN не требует задавать k, автоматически определяет число кластеров
# и помечает выбросы (-1). Полезен для обнаружения аномальных клиентов.
# KMeans: каждому клиенту гарантированно присваивается сегмент.
# Рекомендация: KMeans для сегментации, HDBSCAN для выявления выбросов.
#
# Приоритет импорта:
#   1. sklearn.cluster.HDBSCAN  (sklearn >= 1.3, совместим с numpy 2.x)
#   2. hdbscan (standalone, может конфликтовать с numpy >= 2.0)


def _get_hdbscan(min_cluster_size: int, min_samples: int):  # noqa: ANN202
    """Возвращает HDBSCAN из sklearn (предпочтительно) или standalone."""
    try:
        from sklearn.cluster import HDBSCAN as _HDBSCAN  # sklearn >= 1.3

        return _HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples), "sklearn"
    except ImportError:
        import hdbscan as _hdbscan

        return (
            _hdbscan.HDBSCAN(
                min_cluster_size=min_cluster_size,
                min_samples=min_samples,
                metric="euclidean",
                core_dist_n_jobs=-1,
            ),
            "standalone",
        )


try:
    from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score

    hdb_model, hdb_source = _get_hdbscan(min_cluster_size=50, min_samples=10)
    print(f"Fitting HDBSCAN ({hdb_source})…")
    hdb_labels = hdb_model.fit_predict(X_vals)

    n_hdb = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
    n_noise = int((hdb_labels == -1).sum())
    valid = hdb_labels != -1

    sil_km = silhouette_score(X_vals, km_final.labels_, sample_size=min(10_000, len(X_vals)))
    sil_hdb = (
        silhouette_score(X_vals[valid], hdb_labels[valid], sample_size=min(10_000, valid.sum()))
        if valid.sum() > 1 and len(set(hdb_labels[valid])) > 1
        else float("nan")
    )
    ch_km = calinski_harabasz_score(X_vals, km_final.labels_)
    ch_hdb = (
        calinski_harabasz_score(X_vals[valid], hdb_labels[valid])
        if valid.sum() > 1 and len(set(hdb_labels[valid])) > 1
        else float("nan")
    )
    db_km = davies_bouldin_score(X_vals, km_final.labels_)
    db_hdb = (
        davies_bouldin_score(X_vals[valid], hdb_labels[valid])
        if valid.sum() > 1 and len(set(hdb_labels[valid])) > 1
        else float("nan")
    )

    comp = pd.DataFrame(
        {
            "Метрика": [
                "Кластеров",
                "Выбросов (шум)",
                "Silhouette ↑",
                "Calinski-Harabasz ↑",
                "Davies-Bouldin ↓",
                "Требует задать k",
                "Мягкие границы",
                "Масштаб на prod",
            ],
            f"KMeans (k={best_k})": [
                best_k,
                0,
                f"{sil_km:.4f}",
                f"{ch_km:.1f}",
                f"{db_km:.4f}",
                "Да",
                "Нет",
                "✅ O(n·k·iter)",
            ],
            f"HDBSCAN ({hdb_source})": [
                n_hdb,
                n_noise,
                f"{sil_hdb:.4f}" if not pd.isna(sil_hdb) else "—",
                f"{ch_hdb:.1f}" if not pd.isna(ch_hdb) else "—",
                f"{db_hdb:.4f}" if not pd.isna(db_hdb) else "—",
                "Нет",
                "Да",
                "⚠️  O(n²) память",
            ],
        }
    ).set_index("Метрика")
    print("\n📊 СРАВНЕНИЕ МЕТОДОВ КЛАСТЕРИЗАЦИИ:")
    display(comp)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    km_counts = pd.Series(km_final.labels_).value_counts().sort_index()
    hdb_counts = pd.Series(hdb_labels).value_counts().sort_index()

    axes[0].bar(
        km_counts.index.astype(str),
        km_counts.values,
        edgecolor="black",
        alpha=0.75,
        color="#1f77b4",
    )
    axes[0].set_title(f"KMeans: {best_k} сегментов, 0 выбросов")
    axes[0].set_xlabel("Сегмент")
    axes[0].set_ylabel("Клиентов")

    bar_labels = ["шум" if i == -1 else str(i) for i in hdb_counts.index]
    bar_colors = ["#d62728" if i == -1 else "#2ca02c" for i in hdb_counts.index]
    axes[1].bar(bar_labels, hdb_counts.values, color=bar_colors, edgecolor="black", alpha=0.75)
    axes[1].set_title(f"HDBSCAN: {n_hdb} кластеров + {n_noise} выбросов (красный)")
    axes[1].set_xlabel("Кластер")
    axes[1].set_ylabel("Клиентов")

    plt.suptitle("KMeans vs HDBSCAN: распределение клиентов", fontsize=13)
    plt.tight_layout()
    plt.show()

    winner = "HDBSCAN" if (not pd.isna(sil_hdb) and sil_hdb > sil_km) else f"KMeans (k={best_k})"
    print("\n📌 ВЫВОД:")
    print(f"  • По silhouette лучше: {winner}")
    print(f"  • HDBSCAN выявил {n_noise} клиентов-аномалий ({n_noise / len(X_vals) * 100:.1f}%)")
    print("    — кандидаты для исключения из look-alike эталона")
    print("  • Для production-сегментации рекомендуется KMeans:")
    print("    каждый клиент получает сегмент, O(n) масштабируемость")
    print("  • HDBSCAN — дополнительный фильтр выбросов перед обучением GBM")

except ImportError:
    print("⚠️  HDBSCAN недоступен.")
    print("   sklearn >= 1.3: уже включён  (sklearn.cluster.HDBSCAN)")
    print("   или: uv pip install hdbscan")
except AttributeError as e:
    if "shape" in str(e):
        print("⚠️  Standalone hdbscan несовместим с текущей версией numpy.")
        print("   Обновить sklearn: uv pip install 'scikit-learn>=1.3'")
        print("   sklearn.cluster.HDBSCAN поддерживает numpy 2.x.")
    else:
        raise

In [ ]:
# ruff: noqa: F821
# ── GMM vs KMeans: сравнение с Gaussian Mixture ─────────────────────────────
# GMM допускает мягкие границы (клиент принадлежит кластеру с вероятностью),
# моделирует эллиптические кластеры (не обязательно сферические как KMeans).
try:
    from sklearn.mixture import GaussianMixture

    print(f"Fitting GaussianMixture (k={best_k})...")
    gmm = GaussianMixture(
        n_components=best_k,
        covariance_type="full",
        random_state=42,
        n_init=5,
    )
    gmm_labels = gmm.fit_predict(X_vals)

    n_gmm = len(set(gmm_labels))
    sil_gmm = silhouette_score(X_vals, gmm_labels, sample_size=min(10_000, len(X_vals)))
    ch_gmm = calinski_harabasz_score(X_vals, gmm_labels)
    db_gmm = davies_bouldin_score(X_vals, gmm_labels)

    print(f"GMM: {n_gmm} clusters, silhouette={sil_gmm:.4f}, CH={ch_gmm:.1f}, DB={db_gmm:.4f}")
    print(f"GMM BIC={gmm.bic(X_vals):.1f}, AIC={gmm.aic(X_vals):.1f}")

    # ── Сводная таблица: KMeans vs HDBSCAN vs GMM ────────────────────────────
    # Собираем метрики (sil_km, ch_km, db_km определены в предыдущей ячейке)
    _methods = {
        f"KMeans (k={best_k})": {
            "clusters": best_k,
            "noise": 0,
            "silhouette": sil_km,
            "CH": ch_km,
            "DB": db_km,
        },
        f"GMM (k={best_k})": {
            "clusters": n_gmm,
            "noise": 0,
            "silhouette": sil_gmm,
            "CH": ch_gmm,
            "DB": db_gmm,
        },
    }
    # HDBSCAN results from previous cell (if available)
    try:
        _methods[f"HDBSCAN ({hdb_source})"] = {
            "clusters": n_hdb,
            "noise": n_noise,
            "silhouette": sil_hdb if not pd.isna(sil_hdb) else None,
            "CH": ch_hdb if not pd.isna(ch_hdb) else None,
            "DB": db_hdb if not pd.isna(db_hdb) else None,
        }
    except NameError:
        pass

    comp_rows = []
    for mname, vals in _methods.items():
        comp_rows.append(
            {
                "Метод": mname,
                "Кластеров": vals["clusters"],
                "Выбросов": vals["noise"],
                "Silhouette ↑": f"{vals['silhouette']:.4f}"
                if vals["silhouette"] is not None
                else "—",
                "CH ↑": f"{vals['CH']:.1f}" if vals["CH"] is not None else "—",
                "DB ↓": f"{vals['DB']:.4f}" if vals["DB"] is not None else "—",
            }
        )
    comp_df = pd.DataFrame(comp_rows).set_index("Метод")
    print("\n📊 СРАВНЕНИЕ ТРЁХ МЕТОДОВ КЛАСТЕРИЗАЦИИ:")
    display(comp_df)

    # ── Визуализация ──────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (title, labels) in zip(
        axes,
        [
            (f"KMeans (k={best_k})", km_final.labels_),
            (f"GMM (k={best_k})", gmm_labels),
            ("HDBSCAN", hdb_labels if "hdb_labels" in dir() else None),
        ],
        strict=False,
    ):
        if labels is None:
            ax.text(0.5, 0.5, "HDBSCAN\nне доступен", ha="center", va="center", fontsize=12)
            ax.set_title(title)
            continue
        counts = pd.Series(labels).value_counts().sort_index()
        bar_colors = ["#d62728" if idx == -1 else plt.cm.tab10(idx % 10) for idx in counts.index]
        bar_labels = ["шум" if idx == -1 else str(idx) for idx in counts.index]
        ax.bar(bar_labels, counts.values, color=bar_colors, edgecolor="black", alpha=0.75)
        ax.set_title(title)
        ax.set_xlabel("Кластер")
        ax.set_ylabel("Клиентов")
    plt.suptitle("Распределение клиентов по кластерам: три метода", fontsize=13)
    plt.tight_layout()
    plt.show()

    # ── Вывод ─────────────────────────────────────────────────────────────────
    _best_sil = max(
        (("KMeans", sil_km), ("GMM", sil_gmm)),
        key=lambda x: x[1],
    )
    print("\n📌 ВЫВОД:")
    print(f"  • Лучший silhouette: {_best_sil[0]} ({_best_sil[1]:.4f})")
    print(f"  • GMM BIC={gmm.bic(X_vals):.0f} — чем ниже, тем лучше")
    print("  • KMeans рекомендуется для production: детерминированный, O(n·k)")
    print("  • GMM полезен если кластеры эллиптические (не сферические)")

except ImportError:
    print("⚠️  GaussianMixture недоступен (требуется sklearn)")

In [ ]:
# ruff: noqa: F821
# ── Bootstrap Stability: насколько устойчивы кластеры? ───────────────────────
# Повторно кластеризуем N bootstrap-выборок и измеряем согласованность
# через Adjusted Rand Index (ARI) между парами прогонов.
# ARI = 1.0 → идеально стабильные кластеры, ARI ≈ 0 → случайное совпадение.
from sklearn.metrics import adjusted_rand_score

N_BOOTSTRAP = 50  # Количество bootstrap-итераций
SAMPLE_FRAC = 0.8  # Доля выборки в каждой итерации

print(f"Bootstrap stability test: {N_BOOTSTRAP} iterations, {SAMPLE_FRAC:.0%} sample each...")

rng = np.random.default_rng(42)
bootstrap_labels = []

for i in range(N_BOOTSTRAP):
    idx = rng.choice(len(X_vals), size=int(len(X_vals) * SAMPLE_FRAC), replace=True)
    X_boot = X_vals[idx]
    km_boot = KMeans(n_clusters=best_k, random_state=i, n_init=5)
    labels_boot = km_boot.fit_predict(X_boot)
    bootstrap_labels.append((idx, labels_boot))

# Попарный ARI между прогонами (на пересечении индексов)
ari_scores = []
n_pairs = min(200, N_BOOTSTRAP * (N_BOOTSTRAP - 1) // 2)  # ограничиваем для скорости
pair_count = 0
for i in range(N_BOOTSTRAP):
    for j in range(i + 1, N_BOOTSTRAP):
        if pair_count >= n_pairs:
            break
        idx_i, lab_i = bootstrap_labels[i]
        idx_j, lab_j = bootstrap_labels[j]
        # Находим общие позиции
        common = np.intersect1d(idx_i, idx_j)
        if len(common) < 100:
            continue
        mask_i = np.isin(idx_i, common)
        mask_j = np.isin(idx_j, common)
        # Выравниваем по общим индексам
        _, order_i = np.unique(idx_i[mask_i], return_index=True)
        _, order_j = np.unique(idx_j[mask_j], return_index=True)
        min_len = min(len(order_i), len(order_j))
        if min_len < 50:
            continue
        a = lab_i[mask_i][order_i[:min_len]]
        b = lab_j[mask_j][order_j[:min_len]]
        ari = adjusted_rand_score(a, b)
        ari_scores.append(ari)
        pair_count += 1
    if pair_count >= n_pairs:
        break

ari_arr = np.array(ari_scores)
print(f"\nAdjusted Rand Index (ARI) — {len(ari_scores)} pairs:")
print(f"  Mean ARI: {ari_arr.mean():.3f} ± {ari_arr.std():.3f}")
print(f"  Median ARI: {np.median(ari_arr):.3f}")
print(f"  Min/Max: {ari_arr.min():.3f} / {ari_arr.max():.3f}")

# Визуализация
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(ari_arr, bins=30, edgecolor="black", alpha=0.7, color="#2ca02c")
_mean = ari_arr.mean()
ax.axvline(_mean, color="red", linestyle="--", linewidth=2, label=f"Mean={_mean:.3f}")
ax.set_xlabel("Adjusted Rand Index")
ax.set_ylabel("Frequency")
ax.set_title(f"Bootstrap Stability: ARI distribution ({N_BOOTSTRAP} iterations, k={best_k})")
ax.legend()
plt.tight_layout()
plt.show()

# Интерпретация
if ari_arr.mean() >= 0.80:
    stability = "высокая"
    icon = "✅"
elif ari_arr.mean() >= 0.50:
    stability = "умеренная"
    icon = "⚠️"
else:
    stability = "низкая"
    icon = "❌"
print(f"\n{icon} Стабильность сегментации: {stability} (ARI={ari_arr.mean():.3f})")
print(f"  При повторной кластеризации на 80% данных, {ari_arr.mean():.0%} клиентов")
print("  попадают в тот же сегмент. Порог production: ARI >= 0.70")

In [ ]:
# ruff: noqa: F821
print("=" * 70)
print("ИНТЕРПРЕТАЦИЯ СЕГМЕНТОВ")
print("=" * 70)

KEY_METRICS = [
    "total_amount",
    "tx_count",
    "unique_counterparties",
    "direction_ratio",
    "active_months",
    "amount_growth",
]
key_cols = [c for c in KEY_METRICS if c in segment_profiles.columns]
prof = segment_profiles[key_cols].copy()
overall_med = full_df[key_cols].median()


def _describe_special(val: float, col: str) -> str | None:
    """Возвращает описание для специальных колонок (direction_ratio, amount_growth)."""
    if col == "direction_ratio":
        if val > 0.65:
            return f"▲ {val:.2f} — преимущественно расходы"
        return f"▼ {val:.2f} — входящие" if val < 0.35 else f"→ {val:.2f} — смешанный поток"
    if col == "amount_growth":
        if val > 0.3:
            return f"↑ +{val:.2f} — оборот растёт"
        return f"↓ {val:.2f} — оборот падает" if val < -0.1 else f"→ {val:.2f} — стабильный"
    return None


def describe_vs_median(val: float, med: float, col: str) -> str:
    """Описывает значение метрики относительно медианы."""
    special = _describe_special(val, col)
    if special is not None:
        return special
    ratio = val / med if med != 0 else 1.0
    if ratio > 2.0:
        return f"▲▲ {val:,.1f} (×{ratio:.1f} медианы)"
    if ratio > 1.3:
        return f"▲ {val:,.1f} (выше медианы)"
    if ratio < 0.5:
        return f"▼▼ {val:,.1f} (×{ratio:.1f} медианы)"
    if ratio < 0.8:
        return f"▼ {val:,.1f} (ниже медианы)"
    return f"→ {val:,.1f} (≈ медиана)"


for seg_id in sorted(segment_labels):
    n = (full_df["behavioral_segment"] == seg_id).sum()
    pct = n / len(full_df) * 100
    print(f"\n{'─' * 60}")
    print(f"  Сегмент {seg_id} | {segment_labels[seg_id]}")
    print(f"  Размер: {n:,} клиентов ({pct:.1f}%)")
    print("  Ключевые метрики:")
    for col in key_cols:
        desc = describe_vs_median(prof.loc[seg_id, col], overall_med[col], col)
        print(f"    {col:25s}: {desc}")

print(f"\n{'=' * 70}")
print("ВЫВОД: Для look-alike наиболее ценны клиенты из сегментов с высоким")
print("оборотом и ростом. Малоактивные сегменты — плохие кандидаты для эталона.")
print("=" * 70)

---
## 5. Пересечение с модельным сегментом

Насколько поведенческая сегментация совпадает с формальным модельным сегментом?

In [ ]:
seg_path = os.path.join(OUTPUT_DIR, "segments.parquet")
full_df.to_parquet(seg_path)
print(f"Segments saved: {seg_path}")

# Сохраняем модель KMeans + маппинг меток
model_path = os.path.join(OUTPUT_DIR, "kmeans_model.pkl")
with open(model_path, "wb") as f:
    pickle.dump({"model": km_final, "segment_labels": segment_labels}, f)
print(f"KMeans model + labels saved: {model_path}")

In [ ]:
# Пересечение с ОКВЭД
if "sparkokved_ccode" in full_df.columns:
    okved_cross = pd.crosstab(
        full_df["sparkokved_ccode"].fillna("N/A"),
        full_df["behavioral_segment"],
        margins=True,
        margins_name="Total",
    )
    print("Top-15 OKVED x Behavioral Segment:")
    top_okveds = okved_cross.drop("Total")["Total"].nlargest(15).index
    display(okved_cross.loc[list(top_okveds) + ["Total"]])

---
## 6. Сохранение